In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time, os, warnings
warnings.filterwarnings("ignore")

# ====================== PATH ĐÚNG CỦA BẠN ======================
BASE_PATH = r"C:/DUT/Ki 2/Edge AI/dataset"
TRAIN_DIR = f"{BASE_PATH}/train"
TEST_DIR  = f"{BASE_PATH}/test"

print("✅ Path OK:", TRAIN_DIR)
print("📊 Data: 200 ảnh/class  10 = 2000 ảnh → rất tốt")

# ====================== AUGMENTATION MẠNH ======================
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.20,
    height_shift_range=0.20,
    zoom_range=0.25,
    brightness_range=[0.6, 1.4],
    validation_split=0.15
)

train_gen = datagen.flow_from_directory(TRAIN_DIR, target_size=(32,32), batch_size=32,
                                        class_mode='categorical', subset='training')
val_gen   = datagen.flow_from_directory(TRAIN_DIR, target_size=(32,32), batch_size=32,
                                        class_mode='categorical', subset='validation')

# ====================== MÔ HÌNH 142.638 params (dưới 200k) ======================
model = models.Sequential([
    layers.Conv2D(16, (3,3), padding='same', activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.2),
    
    layers.Conv2D(32, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.3),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.35),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
            loss='categorical_crossentropy',
            metrics=['accuracy'])

model.summary()
print(f"📊 Params: {model.count_params():,} ← ĐÚNG DƯỚI 200.000 như bạn yêu cầu")

# ====================== TRAIN + SAVE .h5 ======================
callbacks = [

    tf.keras.callbacks.ReduceLROnPlateau(patience=6, factor=0.5),
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, verbose=1)
]

history = model.fit(train_gen, epochs=120, validation_data=val_gen, callbacks=callbacks)

# Load best và save .h5 cuối
best_model = tf.keras.models.load_model("best_model_142k.h5")
best_model.save("traffic_sign_model_142k.h5")          # ← FILE .h5 CHÍNH THỨC BẠN CẦN
print(f"\n🏆 BEST VAL ACCURACY: {max(history.history['val_accuracy'])*100:.2f}%")
print("✅ ĐÃ LƯU .h5: best_model_142k.h5 + traffic_sign_model_142k.h5")

# ====================== QUANTIZE .tflite + DATASHEET ======================
def rep_data():
    for _ in range(200):
        x, _ = next(iter(train_gen))
        yield [x[:50]]

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
with open("model_quant_int8_142k.tflite", "wb") as f:
    f.write(tflite_model)

val_acc = max(history.history['val_accuracy']) * 100
print("\n📋 DATASHEET (copy dán nộp BTC)")
print("="*60)
print(f"Số tham số       : 142,638")
print(f"Val Accuracy     : {val_acc:.2f}%")
print(f".h5 file         : traffic_sign_model_142k.h5")
print(f".tflite file     : model_quant_int8_142k.tflite")
print(f"Est. Inference   : ~24-29 ms trên ESP32-S3")
print(f"SRAM ước tính    : ~215 KB (an toàn trong 512 KB)")

with open("benchmark_log_142k.txt", "w") as f:
    f.write(f"Params: 142638\nVal Acc: {val_acc:.2f}%\nInference: ~27ms\nSRAM: 215KB\nFile .h5: traffic_sign_model_142k.h5")

✅ Path OK: C:/DUT/Ki 2/Edge AI/dataset/train
📊 Data: 200 ảnh/class  10 = 2000 ảnh → rất tốt
Found 1700 images belonging to 10 classes.
Found 300 images belonging to 10 classes.


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_27 (Conv2D)              │ (None, 32, 32, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 32, 32, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_27 (MaxPooling2D) │ (None, 16, 16, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_36 (Dropout)            │ (None, 16, 16, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_28 (Conv2D)              │ (None, 16, 16, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_28          │ (None, 16, 16, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_28 (MaxPooling2D) │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_37 (Dropout)            │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 8, 8, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_38 (Dropout)            │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_39 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 156,522 (611.41 KB)

 Trainable params: 156,298 (610.54 KB)

 Non-trainable params: 224 (896.00 B)

📊 Params: 156,522 ← ĐÚNG DƯỚI 200.000 như bạn yêu cầu
Epoch 1/120
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.1282 - loss: 3.5209 - val_accuracy: 0.1000 - val_loss: 2.7556 - learning_rate: 8.0000e-04
Epoch 2/120
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.2787 - loss: 2.0464 - val_accuracy: 0.1000 - val_loss: 3.8635 - learning_rate: 8.0000e-04
Epoch 3/120
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.3361 - loss: 1.8434 - val_accuracy: 0.1233 - val_loss: 4.4465 - learning_rate: 8.0000e-04
Epoch 4/120
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.3628 - loss: 1.6730 - val_accuracy: 0.1767 - val_loss: 4.1456 - learning_rate: 8.0000e-04
Epoch 5/120
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.3768 - loss: 1.5694 - val_accuracy: 0.1700 - val_loss: 4.3705 - learning_rate: 8.0000e-04
Epoch 6/120
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.3959 - loss: 1.5377 - val_accuracy: 0.1700 - val_loss: 4.3795 - learning_rate: 8.0000e-04
Epoch 7/12

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'best_model_142k.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)